# **1. Text Extraction from Case PDF**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !pip install pymupdf python-docx spacy

In [ ]:
import fitz  # PyMuPDF
import re
from docx import Document
import sys
from google.colab import drive, files
# import torch
import os


In [ ]:
# from google.colab import userdata
# userdata.get('GOOGLE_API_KEY')
# userdata.get('HF_TOKEN')


In [ ]:
# 1. Mount Drive
drive.mount('/content/drive')

# 2. Change directory and add paths
os.chdir('/content/drive/MyDrive/LPA_MTech_Project/1_Pre-processing')
sys.path.append('/content/drive/MyDrive/LPA_MTech_Project/2_Evidence_Scrutinization')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def legal_cleaning(text):
    """
    Enhanced cleaning for Indian Law Reports:
    Removes margin ladders (A-H), page headers, and numbers
    while preserving structure for Legal-BERT.
    """
    # 1. Remove "Alphabet Ladders" (Standalone A-H on their own lines)
    # These appear in the margins of court reporters [cite: 1, 6, 209]
    text = re.sub(r'^\s*[A-H]\s*$', '', text, flags=re.MULTILINE)

    # 2. Remove Page Headers/Footers [cite: 109, 110]
    # Targeted patterns found in your specific document
    headers = [
        r'\[2020\]\s*1\s*S\.C\.R\.',
        r'SUPREME\s*COURT\s*REPORTS',
        r'UNION\s*OF\s*INDIA\s*v\.\s*RELIANCE\s*COMMUNICATION.*',
        r'S\.\s*RAVINDRA\s*BHAT,\s*J\.'
    ]
    for pattern in headers:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Remove Standalone Page Numbers [cite: 108, 138, 189]
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Standard Cleaning Logic (Preserved from your original code)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r' +', ' ', text)
    text = "".join(char for char in text if char.isprintable() or char in ['\n', '\t'])

    # Final trim to ensure no trailing blank lines from the removed ladders
    text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])

    return text.strip()

In [ ]:
def process_document():
    # 2. Colab-Native File Selection
    print("Please upload your Legal Document (PDF or Word):")
    uploaded = files.upload() # This replaces the tkinter/filedialog logic

    if not uploaded:
        print("Selection cancelled.")
        return None

    # Get the name of the uploaded file
    file_path = list(uploaded.keys())[0]
    ext = os.path.splitext(file_path)[1].lower()

    print(f"Extracting: {file_path}...")

    # Get the file name and remove extension from the filename
    fileName = os.path.splitext(file_path)[0]

    # 3. Text Extraction
    raw_text = ""
    try:
        if ext == ".pdf":
            with fitz.open(file_path) as doc:
                raw_text = "\n".join([page.get_text("text") for page in doc])
        elif ext == ".docx":
            doc = Document(file_path)
            raw_text = "\n".join([para.text for para in doc.paragraphs])

        # 4. Pre-processing & Cleaning
        # Ensure the legal_cleaning function is defined in a previous cell
        cleaned_text = legal_cleaning(raw_text)

        # 5. Output for later stages - Saving to your mounted Drive
        # Using the path we established in your LPA_PART2 folder structure
        folder_name = "/content/drive/MyDrive/LPA_MTech_Project/1_Pre-processing/Text_Extracts"
        os.makedirs(folder_name, exist_ok=True)
        output_file = os.path.join(folder_name, f"{fileName}.txt")

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"Success! Cleaned text saved to Drive: {output_file}")
        print(f"Character count: {len(cleaned_text)}")
        return cleaned_text

    except Exception as e:
        print(f"Extraction error: {str(e)}")
        return None

In [ ]:
cleaned_data = process_document()


Please upload your Legal Document (PDF or Word):


Saving 2020_1_41_53_EN.pdf to 2020_1_41_53_EN.pdf
Extracting: 2020_1_41_53_EN.pdf...
Success! Cleaned text saved to Drive: /content/drive/MyDrive/LPA_MTech_Project/1_Pre-processing/Text_Extracts/2020_1_41_53_EN.txt
Character count: 27386


# **1.5. The Segmentation method for extracting Evidence, Facts and IPCs from cleaned Text file**

In [ ]:
import json
from google import genai
from google.genai import types
from google.colab import userdata
import os

# Initialize the Client directly using the Colab Secret
# This replaces the os.environ["GOOGLE_API_KEY"] call
api_key = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=api_key)

def extract_legal_details(legal_text):
    prompt = f"""
    You are a Legal Expert in Indian Law. Analyze the following court text and extract:
    1. FACTS: The core background of the dispute.
    2. EVIDENCE: Specific documents, testimonies, or proofs cited.
    3. IPC/STATUTES: Any Indian Penal Code sections or specific Acts mentioned.

    Format the output as a clean Python Dictionary not a list.

    TEXT:
    {legal_text}
    """

    # Use the new models.generate_content method
    response = client.models.generate_content(
        model='gemini-3-flash-preview',
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level="MEDIUM"),
            response_mime_type="application/json",
        )
    )


    try:
        data = json.loads(response.text)
        return data
    except json.JSONDecodeError:
        print("Failed to parse JSON")
        return None

In [ ]:
# Get the extracted facts, evidence and basic IPCs from the text file
result = extract_legal_details(cleaned_data)
print(result)

{'FACTS': "Moola Gounder (MG) formed a Mitakshara coparcenary with his two sons, Palanisamy (D1) and Arumugam (D2). MG died intestate in 1971, leaving behind his wife (D5), two sons, and three daughters (including the minor Plaintiff). Upon his death, his 1/3 share in the coparcenary devolved to his legal heirs. In 1973, a registered release deed was executed by the mother (D5) on behalf of herself and the minor plaintiff, relinquishing their rights in the property to the sons. Subsequently, in 1980, a registered partition deed was executed between the two sons, witnessed by the plaintiff's husband. In 1989, the plaintiff filed a suit for partition, claiming the 1973 release deed was void ab initio as the mother could not act as a natural guardian for joint family property and had no court sanction to relinquish a minor's interest.", 'EVIDENCE': "Registered release deed dated 10.03.1973 executed by the mother (D5) and daughters; Registered partition deed dated 24.04.1980 between defend

In [ ]:
print(f"The processed facts using gemini 3 flash are:")
case_facts = result.get('FACTS')
print(case_facts)

The processed facts using gemini 3 flash are:
Moola Gounder (MG) formed a Mitakshara coparcenary with his two sons, Palanisamy (D1) and Arumugam (D2). MG died intestate in 1971, leaving behind his wife (D5), two sons, and three daughters (including the minor Plaintiff). Upon his death, his 1/3 share in the coparcenary devolved to his legal heirs. In 1973, a registered release deed was executed by the mother (D5) on behalf of herself and the minor plaintiff, relinquishing their rights in the property to the sons. Subsequently, in 1980, a registered partition deed was executed between the two sons, witnessed by the plaintiff's husband. In 1989, the plaintiff filed a suit for partition, claiming the 1973 release deed was void ab initio as the mother could not act as a natural guardian for joint family property and had no court sanction to relinquish a minor's interest.


In [ ]:
print(f"The processed evidence using gemini 3 flash are:")
case_evidence = result.get('EVIDENCE')
print(case_evidence)

The processed evidence using gemini 3 flash are:
Registered release deed dated 10.03.1973 executed by the mother (D5) and daughters; Registered partition deed dated 24.04.1980 between defendant nos. 1 and 2; Registered settlement deed dated 30.04.1980 executed by the sons in favor of the mother; Attestation of the 1980 deeds by the plaintiff's husband as a witness; Records of subsequent property transfers and possession by defendant nos. 1 and 2.


In [ ]:
print(f"The processed IPCs using gemini 3 flash are:")
case_ipc = result.get('IPC/STATUTES')
print(case_ipc)

The processed IPCs using gemini 3 flash are:
None


# **2. Load the Evidence Scrutinization Llama 3 model**

In [ ]:
# Load model for 1st time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = "unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit"
adapter_id = "Raazi29/Nyaya-Llama-3.1-8B-Indian-Legal"

# Use bitsandbytes directly
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16 # T4 supports this, keeps it snappy
)

model = PeftModel.from_pretrained(model, adapter_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/671M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

In [ ]:
# # Save the model
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("/content/drive/MyDrive/LPA_MTech_Project/Nyaya-Llama-Merged")
# tokenizer.save_pretrained("/content/drive/MyDrive/LPA_MTech_Project/Nyaya-Llama-Merged")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


('/content/drive/MyDrive/LPA_MTech_Project/Nyaya-Llama-Merged/tokenizer_config.json',
 '/content/drive/MyDrive/LPA_MTech_Project/Nyaya-Llama-Merged/special_tokens_map.json',
 '/content/drive/MyDrive/LPA_MTech_Project/Nyaya-Llama-Merged/chat_template.jinja',
 '/content/drive/MyDrive/LPA_MTech_Project/Nyaya-Llama-Merged/tokenizer.json')

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch

# # Load directly from your Drive folder
# path = "/content/drive/MyDrive/LPA_MTech_Project/Nyaya-Llama-Merged"
# model = AutoModelForCausalLM.from_pretrained(
#     path,
#     device_map="auto",
#     torch_dtype=torch.bfloat16
# )
# tokenizer = AutoTokenizer.from_pretrained(path)

In [ ]:
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Memory allocated: 6.20 GB
Memory reserved: 7.34 GB


In [ ]:
prompt_template = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a legal analyst. Scrutinize the provided EVIDENCE against the established FACTS.
Identify contradictions, timeline gaps, or logical inconsistencies.
Be concise and objective.<|eot_id|>

<|start_header_id|>user<|end_header_id|>
### ESTABLISHED FACTS:
{case_facts}

### DISPUTED EVIDENCE:
{case_evidence}

### ANALYSIS:
<|assistant|><|end_header_id|>
"""

In [ ]:
# 1. Tokenize the input and move to GPU
inputs = tokenizer(prompt_template, return_tensors="pt").to("cuda")

# 2. Generate the response
# We use a low temperature for objective, legal analysis
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# 3. Decode and print only the new analysis text
# We slice the output to remove the original prompt from the display
response = tokenizer.decode(output_tokens[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print("--- LEGAL ANALYSIS ---")
print(response)

--- LEGAL ANALYSIS ---
**Contradictions and Inconsistencies:**

1.  **Timeline Inconsistency**: The release deed (1973) and partition deed (1980) were executed 7 years apart. This gap raises questions about the necessity of the release deed, as the partition deed was executed later, which could have potentially included the minor plaintiff's interest.
2.  **Lack of Court Sanction**: The mother (D5) executed the release deed on behalf of the minor plaintiff without court sanction, which is a legal requirement for minors. This makes the release deed potentially void ab initio.
3.  **Attestation by Plaintiff's Husband**: The plaintiff's husband attested the partition deed (1980) as a witness. However, this raises questions about his impartiality, as he is a party to the transaction (defendant no. 2).
4.  **Settlement Deed**: The settlement deed (1980) executed by the sons in favor of the mother seems to contradict the partition deed, as it suggests that the mother was still involved in th

In [ ]:
# scr_result = scrutinize_legal_data(case_facts, case_evidence, case_ipc)